# 03 — Link prediction sanity

Two complementary checks:

1. **Pragmatic (engineering) baseline:** Euclidean distance on a **hand-built** vector: normalized direction plus **log1p** of `Inf.Hyp.Rad` and `Inf.Kappa`. This is cheap and can correlate with edges, but it is **not** the PS model’s native geometry and it mixes scales heuristically.

2. **Native-style hyperbolic distance:** Geodesic length on the **Lorentz hyperboloid** built only from **unit direction + `Inf.Hyp.Rad`** (see `native_hyperbolic.py`). This matches standard **H^{D+1}** geodesics under the usual ρ–ŝ chart. It still ignores **κ** (κ shapes **connection probabilities**, not point–point geodesic length in this chart).

Assumptions for (2) are spelled out in the markdown before that code cell; if the export’s radial parameter differs slightly from ρ in the paper, treat the Lorentz curve as a structured baseline rather than a certified replica of internal D-Mercator distances.


### Pragmatic baseline — Euclidean distance in a hand-built feature space

**Method:** Each vertex is represented by **[unit direction | log1p(Inf.Hyp.Rad) | log1p(Inf.Kappa)]** (concatenated). Euclidean distance is computed for random **edges** and random **non-edges**. Vertex keys are normalized with `str(...)` so graph and table align.

**Why “pragmatic”?** We are **not** using the model’s Fermi–Dirac / hyperbolic likelihood—only a quick vector space proxy so you can eyeball whether edges look “closer” than random pairs in a comparable feature scale.

**How to read the output:** If edge distances are systematically **smaller** than non-edge distances, the feature space carries some link information (a weak baseline). Overlap of histograms means this metric is not separable. Compare qualitatively with the **Lorentz** histogram below (different units on the x-axis).


In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

import dmercator3d_io as dm

RUN_SUBDIR = os.environ.get("DMERCATOR_RUN", "d3")
paths = dm.paths_for_run(RUN_SUBDIR)
_, df = dm.parse_inf_coord(paths["inf_coord"])
G = dm.load_edges_graph(paths["edge"])
pos_idx = {str(v): i for i, v in enumerate(df["Vertex"])}
U = dm.normalize_direction_nd(df)
hyp = np.log1p(df["Inf.Hyp.Rad"].to_numpy())
kap = np.log1p(df["Inf.Kappa"].to_numpy())
X = np.hstack([U, hyp[:, None], kap[:, None]])


def dist(i, j):
    d = X[i] - X[j]
    return float(np.sqrt(np.dot(d, d)))


rng = np.random.default_rng(2)
verts = list(G.nodes())
edges = list(G.edges())
n_pos = min(5000, len(edges))
n_neg = min(5000, len(edges))
pos_pairs = [edges[i] for i in rng.choice(len(edges), size=n_pos, replace=False)]
edge_set = set(tuple(sorted(e)) for e in G.edges())
neg = []
while len(neg) < n_neg:
    a, b = rng.choice(verts, size=2, replace=False)
    if tuple(sorted((a, b))) in edge_set:
        continue
    neg.append((a, b))

de = [dist(pos_idx[str(a)], pos_idx[str(b)]) for a, b in pos_pairs if str(a) in pos_idx and str(b) in pos_idx]
dn = [dist(pos_idx[str(a)], pos_idx[str(b)]) for a, b in neg if str(a) in pos_idx and str(b) in pos_idx]
plt.figure(figsize=(6, 3))
plt.hist(de, bins=50, alpha=0.6, label="edges", density=True)
plt.hist(dn, bins=50, alpha=0.6, label="non-edges", density=True)
plt.legend()
plt.xlabel("Euclidean dist (feature space)")
plt.tight_layout()
plt.show()


### Native-style check — Lorentz hyperboloid geodesic distance

**Method:** From each row, form a point on the **upper Lorentz sheet** in ℝ^{D+2}: `X = (cosh ρ, sinh ρ · ŝ)` with ŝ the **L2-normalized** `Inf.Pos.*` vector and **ρ = Inf.Hyp.Rad** (interpreted as hyperbolic radial coordinate in that chart). Pairwise geodesic distance is `d(u,v) = acosh(clamp(−⟨u,v⟩, min=1+ε))` with Minkowski inner product `−u₀v₀ + Σ uᵢvᵢ` (see `native_hyperbolic.py`).

**Caveats:** (1) **κ is omitted**—the paper’s link model uses κ in the Fermi–Dirac term; geodesic length alone does not reproduce connection logits. (2) If the inference program’s radial variable differs by an affine or curvature rescaling from this textbook ρ, distances are off by a monotone map; compare **edge vs non-edge** shape, not absolute numbers to other papers. (3) Optional global scale: multiply by `R` if you need curvature **−1/R²** (see module docstring re `radius_H^D+1`).

**How to read the output:** Same random edge / non-edge pairs as above (`random_state=2`). Histograms use **the same** pair lists so differences between the two figures are due to **metric choice**, not sampling. Edges often sit at slightly smaller **geodesic** distance when the embedding is informative; heavy overlap is still common for sparse PPIs.

In [ ]:
# Same subsample as previous cell (rebuild with identical RNG so histograms are comparable)
from native_hyperbolic import directions_hyp_to_lorentz, lorentz_geodesic_pairwise

rho = df["Inf.Hyp.Rad"].to_numpy(dtype=np.float64)
X_lor = directions_hyp_to_lorentz(U, rho)

rng = np.random.default_rng(2)
verts = list(G.nodes())
edges = list(G.edges())
n_pos = min(5000, len(edges))
n_neg = min(5000, len(edges))
pos_pairs = [edges[i] for i in rng.choice(len(edges), size=n_pos, replace=False)]
edge_set = set(tuple(sorted(e)) for e in G.edges())
neg = []
while len(neg) < n_neg:
    a, b = rng.choice(verts, size=2, replace=False)
    if tuple(sorted((a, b))) in edge_set:
        continue
    neg.append((a, b))


def pair_indices(pairs):
    ia, ib = [], []
    for a, b in pairs:
        sa, sb = str(a), str(b)
        if sa in pos_idx and sb in pos_idx:
            ia.append(pos_idx[sa])
            ib.append(pos_idx[sb])
    return np.asarray(ia, dtype=int), np.asarray(ib, dtype=int)


ia_e, ib_e = pair_indices(pos_pairs)
ia_n, ib_n = pair_indices(neg)
de_lor = lorentz_geodesic_pairwise(X_lor, ia_e, ib_e)
dn_lor = lorentz_geodesic_pairwise(X_lor, ia_n, ib_n)

plt.figure(figsize=(6, 3))
plt.hist(de_lor, bins=50, alpha=0.6, label="edges", density=True)
plt.hist(dn_lor, bins=50, alpha=0.6, label="non-edges", density=True)
plt.legend()
plt.xlabel("Lorentz hyperboloid geodesic distance (unit curvature)")
plt.tight_layout()
plt.show()